# MaxText with GEPA Optimization

## Overview
This notebook demonstrates how to use **GEPA** (Generic Evaluation and Prompt Adaptation) to optimize system prompts for MaxText models. It shows how to serve a model using MaxText/vLLM on a TPU, and use GEPA to find a better prompt based on evaluation results.

This tutorial was designed to be run on a single **TPU GCE VM** using **VS Code** or **JupyterLab**.

## Installation: MaxText and Post training Dependencies

Before proceeding, create a virtual environment and install the required post-training dependencies by following `Option 3: Installing [tpu-post-train]` in the [MaxText installation guide](https://maxtext.readthedocs.io/en/latest/install_maxtext.html#from-source). Once the environment is set up, ensure the notebook is running within it.

In [ ]:
import os
import re
import socket
import subprocess
import sys
import time
import random
import gepa
from datasets import load_dataset
from getpass import getpass
from maxtext.utils.globals import MAXTEXT_PKG_DIR
from gepa.adapters.default_adapter.default_adapter import DefaultAdapter
from gepa.core.adapter import EvaluationBatch

# Change directory to maxtext root
%cd ~/maxtext

# Setup API tokens
if "HF_TOKEN" not in os.environ:
    os.environ["HF_TOKEN"] = getpass("Enter your HuggingFace Token (HF_TOKEN): ")

if "GEMINI_API_KEY" not in os.environ:
    os.environ["GEMINI_API_KEY"] = getpass("Enter your Gemini API Key (GEMINI_API_KEY): ")

HF_TOKEN = os.environ["HF_TOKEN"]
GEMINI_API_KEY = os.environ["GEMINI_API_KEY"]

# Model configurations
MODEL_NAME = "qwen3-4b"
TOKENIZER_NAME = "Qwen/Qwen3-4B-Instruct-2507" 
MODEL_CHECKPOINT_PATH = f"{MAXTEXT_PKG_DIR}/qwen3_4b_checkpoint"
LOAD_PARAMETERS_PATH = f"{MODEL_CHECKPOINT_PATH}/0/items"

print(f"Set up complete. Working directory: {os.getcwd()}")

## Download and Convert Checkpoint

In [ ]:
LOCAL_HF_DIR = os.path.expanduser("~/maxtext/local_hf_weights")

os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

print(f"Downloading {TOKENIZER_NAME} via huggingface-cli to {LOCAL_HF_DIR}...")

# Download the weights via huggingface-cli
os.makedirs(LOCAL_HF_DIR, exist_ok=True)
subprocess.run([
    "huggingface-cli", "download", TOKENIZER_NAME,
    "--local-dir", LOCAL_HF_DIR,
    "--local-dir-use-symlinks", "False",
    "--include", "*.safetensors", "*.json", "*.txt", "*.model", "*.jinja"
], check=True)

print("Starting checkpoint conversion...")

env = os.environ.copy()
env["JAX_PLATFORMS"] = "cpu"
env["PYTHONPATH"] = MAXTEXT_PKG_DIR

# Convert the local Hugging Face checkpoint to MaxText/Orbax format
cmd = [
    sys.executable, "-m", "maxtext.checkpoint_conversion.to_maxtext",
    f"model_name={MODEL_NAME}",
    f"base_output_directory={MODEL_CHECKPOINT_PATH}",
    f"--hf_model_path={LOCAL_HF_DIR}",
    f"hf_access_token={HF_TOKEN}",
    "hardware=cpu",
    "use_multimodal=false",
    "scan_layers=false",
    "skip_jax_distributed_system=True",
    "--lazy_load_tensors=True", 
]

subprocess.run(cmd, env=env, check=True)
print("Conversion complete!")


# Start vLLM Server

In [ ]:
def is_server_running(port=8000):
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        return s.connect_ex(('localhost', port)) == 0

def kill_existing_server(port=8000):
    if is_server_running(port):
        print(f"Port {port} is in use. Killing the main server process...")
        subprocess.run(f"fuser -k {port}/tcp", shell=True, stderr=subprocess.DEVNULL, stdout=subprocess.DEVNULL)
        time.sleep(3) 
        if is_server_running(port):
            print(f"Force killing process on port {port}...")
            subprocess.run(f"fuser -k -9 {port}/tcp", shell=True, stderr=subprocess.DEVNULL, stdout=subprocess.DEVNULL)
            subprocess.run(f"lsof -t -i:{port} | xargs -r kill -9", shell=True, stderr=subprocess.DEVNULL, stdout=subprocess.DEVNULL)
            time.sleep(3)
            
    print("Cleaning up any orphaned vLLM or Ray worker processes...")
    subprocess.run("pkill -9 -f vllm", shell=True, stderr=subprocess.DEVNULL, stdout=subprocess.DEVNULL)
    subprocess.run("pkill -9 -f ray", shell=True, stderr=subprocess.DEVNULL, stdout=subprocess.DEVNULL)
    
    print("Releasing TPU hardware locks...")
    subprocess.run("fuser -k -9 /dev/accel*", shell=True, stderr=subprocess.DEVNULL, stdout=subprocess.DEVNULL)
    subprocess.run("fuser -k -9 /dev/tpu*", shell=True, stderr=subprocess.DEVNULL, stdout=subprocess.DEVNULL)
    
    time.sleep(2)


# Command to start vLLM server on TPU (using the MaxText model in vLLM out-of-tree)
server_cmd = f"""
NEW_MODEL_DESIGN=1 SKIP_JAX_PRECOMPILE=1 vllm serve {TOKENIZER_NAME} \\
    --tensor-parallel-size 4 \\
    --max-model-len 16384 \\
    --gpu-memory-utilization 0.8 \\
    --hf-overrides '{{"architectures": ["MaxTextForCausalLM"]}}' \\
    --additional-config '{{"maxtext_config": {{"model_name": "{MODEL_NAME}", "log_config": false, "load_parameters_path": "{LOAD_PARAMETERS_PATH}", "enable_dp_attention": true}}, "sharding": {{"sharding_strategy": {{"enable_dp_attention": true}}}}}}' \\
    --port 8000 > ../vllm_server.log 2>&1
"""

kill_existing_server(8000)

print("Starting vLLM Server on the TPU...")
process = subprocess.Popen(server_cmd, shell=True)

print("Waiting for server to initialize and compile model...")
time.sleep(90) 
print("Server initialization period finished. Moving to GEPA optimization.")

# Display server logs
! cat ../vllm_server.log

## GEPA: Prompt Optimization

In [ ]:
def init_dataset():
    print(f"Streaming NuminaMath dataset until we find 600 AMC/AIME problems...")
    
    # Use streaming=True to completely bypass the 30+ minute download!
    dataset = load_dataset("AI-MO/NuminaMath-CoT", split="train", streaming=True)
    
    all_examples = []
    
    # Process items over the network one by one
    for x in dataset:
        if x["source"] == "amc_aime":
            solution = x["solution"]
            
            match = re.search(r'\\boxed\{([^}]+)\}', solution)
            if match:
                final_answer = match.group(1).strip()
                
                if final_answer.replace('.', '', 1).replace('-', '', 1).isdigit():
                    all_examples.append({
                        "input": x["problem"], 
                        "additional_context": {"solution": solution}, 
                        "answer": final_answer
                    })

        # Stop fetching from the internet once we have 600 examples
        if len(all_examples) >= 600:
            break

    random.Random(42).shuffle(all_examples)

    trainset = all_examples[0:200]
    valset = all_examples[200:500]    
    testset = all_examples[500:600]           

    print(f"Loaded {len(trainset)} train, {len(valset)} val, and {len(testset)} test examples.")
    return trainset, valset, testset

trainset, valset, testset = init_dataset()

**Robust Answer Extraction**

In mathematical competition problems (like AMC or AIME), the model is instructed to provide its final result inside the LaTeX command `\boxed{}`. 

However, simple extraction methods (like basic regular expressions) often fail when the answer itself contains nested braces, which is common in LaTeX expressions such as fractions like `$\frac{1}{2}$`. A simple search for the first closing brace `}` would terminate prematurely, capturing only part of the answer. 

The `extract_math_answer` function below ensures that the entire content within the correct matching closing brace is retrieved by counting nested braces.

In [ ]:
def extract_math_answer(prediction):
    """
    Robustly extracts the content inside \boxed{} by counting nested braces.
    This prevents failures when the answer contains LaTeX like \frac{1}{2}.
    """
    # Find where the \boxed{ starts
    match = re.search(r'\\boxed\{', prediction)
    if not match:
        return ""
    
    start_idx = match.end()
    brace_count = 1
    
    # Iterate through characters to find the matching closing brace
    for i, char in enumerate(prediction[start_idx:]):
        if char == '{':
            brace_count += 1
        elif char == '}':
            brace_count -= 1
        
        if brace_count == 0:
            return prediction[start_idx:start_idx+i].strip()
            
    return "" # Returns empty if braces are unbalanced

class MathAdapter(DefaultAdapter):
    def evaluate(self, batch, candidate, capture_traces=False):
        # Track prompt evolution
        print("--- Evaluating Candidate Prompt (truncated) ---")
        print(candidate.get("system_prompt", "")[:200] + "...")
        print("---------------------------------------------")
        
        eval_batch = super().evaluate(batch, candidate, capture_traces)
        
        new_scores = []
        updated_trajectories = list(eval_batch.trajectories) if eval_batch.trajectories else [{} for _ in range(len(batch))]
        
        for i, (data, output) in enumerate(zip(batch, eval_batch.outputs)):
            prediction = output.get("full_assistant_response", "")
            ground_truth = str(data["answer"]).strip()
            
            extracted_answer = extract_math_answer(prediction)

            try:
                score = 1.0 if float(extracted_answer) == float(ground_truth) else 0.0
            except ValueError:
                score = 0.0 
                
            new_scores.append(score)
            
            # Add further debugging lines
            print(f"\n[Debug Eval] Item {i}:")
            print(f"  Problem: {data['input'][:50]}...")
            print(f"  Ground Truth: {ground_truth}")
            print(f"  Extracted Answer: {extracted_answer}")
            print(f"  Score: {score}")
            
            if capture_traces:
                updated_trajectories[i]["extracted_answer"] = extracted_answer
            
        return EvaluationBatch(
            outputs=eval_batch.outputs,
            scores=new_scores,
            trajectories=updated_trajectories
        )

In [ ]:
os.environ["OPENAI_API_BASE"] = "http://localhost:8000/v1"
os.environ["OPENAI_API_KEY"] = "fake-key"

seed_prompt = {
    "system_prompt": "You are a mathematical reasoning engine. Solve the following problem "
        "step-by-step. Keep your reasoning concise and strictly avoid going into infinite loops. "
        "Provide the final numerical result inside \\boxed{}."
}

adapter = MathAdapter(
    model=f"openai/{TOKENIZER_NAME}",
    max_litellm_workers=2,
    litellm_batch_completion_kwargs={
        "max_tokens": 8192, 
        "temperature": 0.0
    }
)

print("Starting GEPA optimization...")
result = gepa.optimize(
    seed_candidate=seed_prompt,
    trainset=trainset,
    valset=valset,
    adapter=adapter,
    max_metric_calls=1000,
    reflection_lm="gemini/gemini-3-flash-preview",
    display_progress_bar=False,
    reflection_minibatch_size=8,
    skip_perfect_score=True,
    use_merge=True,
    max_merge_invocations=5,
    merge_val_overlap_floor=5
)

print("Optimized prompt:", result.best_candidate['system_prompt'])

In [ ]:
print("Evaluating the original seed prompt on the validation set...")
before_eval = adapter.evaluate(testset, seed_prompt)
before_correct = sum(before_eval.scores)
before_total = len(before_eval.scores)
print(f"Accuracy before optimization: {before_correct / before_total * 100:.1f}% ({before_correct}/{before_total})")

print("\nEvaluating the optimized prompt on the validation set...")
after_eval = adapter.evaluate(testset, result.best_candidate)
after_correct = sum(after_eval.scores)
after_total = len(after_eval.scores)
print(f"Accuracy after optimization: {after_correct / after_total * 100:.1f}% ({after_correct}/{after_total})")

print("\n" + "="*50)
print("FINAL OPTIMIZED PROMPT:")
print("="*50)
print(result.best_candidate['system_prompt'])

### Case Study: Large-Scale AIME Prompt Progression

While the cells above demonstrate the mechanics of GEPA on a small validation set, running this optimization at scale yields profound changes in the model's behavior. This report details how the system prompt evolved during a large-scale GEPA optimization process for the AIME dataset, leading to an overall improvement from **49.0% to 54.0%** in test accuracy.

#### Iteration 0: Baseline
* **Test Accuracy:** 49.0%
* **Characteristics:** The seed prompt was a simple instruction to solve math problems step-by-step and provide the answer in `\boxed{}`.
* **Key Challenge:** AIME problems require deep mathematical knowledge and complex chain-of-thought reasoning, making it much harder than standard benchmarks like GSM8K.

#### Iteration 2: Injecting Domain Knowledge
As GEPA reflected on failure cases, it began adding specific mathematical facts to the prompt to help the model:
* **Combinatorial Game Theory (Chomp):** Referenced the Stars and Bars formula $\binom{n+m}{m}$.
* **3D Lattice Geometry:** Stated the number of equilateral triangles in a $3 \times 3 \times 3$ grid.
* **Cube Properties:** Reminded the model of edge and vertex counts.

#### Iteration 3: Advanced Heuristics
GEPA continued to add highly specific problem-solving strategies to address recurring errors:
* **Permutations with Bounded Displacement:** Noted the Fibonacci recurrence.
* **Circle Packing:** Provided formulas for staggered rows.
* **Golden Ratio:** Added key identities.
* **Gas Mileage Logic:** Included specific tips for fuel consumption word problems.

#### Iteration 5: Comprehensive Cheat Sheet
Eventually, GEPA consolidated this knowledge into structured heuristics for specific domains:
* **Geometry:** Circles in a wedge follow a geometric progression.
* **Number Theory:** Permutations and squares requiring the same square-free part.
* **Combinatorics:** Subset sums handling disjoint vs. overlapping pairs.

#### Key Code Components
The success of this optimization relied heavily on the evaluation pipeline defined earlier in this notebook:
* `MathAdapter`: The GEPA adapter class handling candidate evaluation and metric tracking.
* `extract_math_answer`: A robust helper function using brace counting to handle complex LaTeX in boxed answers without failing on nested braces.

#### Final Verdict
The optimized prompt effectively acted as a rich mathematical cheat sheet, achieving **54.0% accuracy** on the full test set—a solid **5.0% absolute improvement** over the baseline prompt.

### Final optimized prompt from the study:
```
You are a mathematical reasoning engine. Your task is to solve complex, competition-level mathematical problems (such as those from AMC, AIME, or Olympiads) step-by-step. Provide concise reasoning and ensure the final numerical result is placed inside \boxed{}.

### General Instructions:
- **Conciseness**: Avoid unnecessary repetition or verbose explanations. Get straight to the mathematical core.
- **Strict Logic**: Follow a logical progression from the given information to the final result.
- **Infinite Loops**: Avoid re-calculating the same values without progress; if a strategy fails, pivot to a new mathematical approach.

### Domain-Specific Heuristics and Strategies:
1. **Geometry - Circles in a Wedge**: 
   - For a chain of circles tangent to two intersecting lines and tangent to each other consecutively, the radii follow a **geometric progression**. 
   - The radius of a middle circle in a sequence is the geometric mean of the radii of the surrounding circles (e.g., $r_3 = \sqrt{r_1 \cdot r_5}$ for a 5-circle chain).
2. **Number Theory - Permutations and Squares**:
   - If a problem requires $k \cdot a_k$ to be a perfect square, recognize that $k$ and $a_k$ must have the same **square-free part** (the radical).
   - Group numbers into equivalence classes based on their square-free parts. The number of such permutations is the product of the factorials of the sizes of these classes: $P(n) = \prod |C_t|!$.
3. **Combinatorics - Subset Sums**:
   - When counting subsets where pairs of elements must sum to specific values, consider two distinct cases:
     - **Case 1 (Disjoint Pairs)**: The pairs satisfying the different sum conditions share no elements.
     - **Case 2 (Overlapping/Linked Pairs)**: The pairs share an element (e.g., $a+b=S_1$ and $b+c=S_2$), forming a triple or larger chain. Ensure you account for both cases and subtract any double-counted subsets.
4. **Algebra - Sum of Square Roots**:
   - If $\sqrt{x} + \sqrt{y} = \sqrt{N}$ for integers $x, y, N$, then $xy$ must be a perfect square.
   - Always verify constraints regarding $N$ (e.g., if the problem states $N$ is not a perfect square, exclude cases where $\sqrt{x} + \sqrt{y}$ results in an integer).
5. **Divisibility and Quotients/Remainders**:
   - For problems involving $n = 100q + r$ (or other bases), identify the valid ranges for $q$ and $r$. 
   - When counting $q+r \equiv 0 \pmod{k}$, determine the number of values in the range of $r$ that fall into each residue class modulo $k$ to accurately sum the possibilities across all $q$.
6. **Coordinate Geometry - Parabolas**:
   - Use the focus-directrix definition: $\sqrt{(x-x_f)^2 + (y-y_f)^2} = \frac{|ax+by+c|}{\sqrt{a^2+b^2}}$.
   - Simplify the resulting quadratic equation to look for patterns like $(Ax+By)^2 + Dx + Ey + F = 0$.
   - For integer point counts, transform the equation into a linear Diophantine form for specific values of $(Ax+By)$.

### Final Formatting:
Place the final numerical answer or choice (e.g., \boxed{12} or \boxed{210}) at the end of your response.
```


## Inference with Optimized Prompt

In [ ]:
from openai import OpenAI

# Fallback to a default prompt if optimization was not run in this session
try:
    optimized_prompt = result.best_candidate['system_prompt']
except NameError:
    print("Warning: Optimization result not found. Using a default system prompt.")
    optimized_prompt = seed_prompt["system_prompt"]

math_problem = "Solve for x: 2x + 5 = 15"

client = OpenAI(base_url="http://localhost:8000/v1", api_key="not-needed")

print("Sending request to the running vLLM Server...")

response = client.chat.completions.create(
  model=TOKENIZER_NAME,
  messages=[
    {"role": "system", "content": optimized_prompt},
    {"role": "user", "content": math_problem}
  ],
  max_tokens=1024,
  temperature=0.0  # Greedy decoding for deterministic answers
)

print("\n=== Model Output ===")
print(response.choices[0].message.content)